# extension_2024 / e1: Data Construction and Cleaning Pipeline (extended to 2024)

Forked from `code/v2/1_cleaning_master_data_FINAL.ipynb`. Changes:

1. Year range bumped from 1995-2019 to 1995-2024.
2. ECI source switched to the local Atlas file `intermediary/rawdata/growth_proj_eci_rankings-1.csv`. Both `eci_hs92` and Atlas's own `growth_proj` are retained.
3. PWT 11.0 still hard-stops at 2019; HCI will be NaN for 2020-2024. Flagged in coverage diagnostic at the end.
4. IMF WEO bumped from April 2024 to April 2025 vintage with fallback.
5. NR production cell now reuses the existing cache by default; bypass with FORCE_REFRESH.
6. A coverage diagnostic cell is added at the end reporting non-null counts per variable per year.

Outputs:
- `intermediary/master_data_long.csv` (long format)
- `intermediary/master_data_wide.csv` (wide format)
- `intermediary/Master_extended.csv` (headline panel, downstream consumers point here)


## 0. Setup

In [28]:
import sys, os
# The notebook's working directory is extension_2024/. _config and
# standardize_country (the local shim) live alongside this notebook.
import _config as cfg
from standardize_country import add_iso3, ALIAS_TO_ISO3, ISO3_TO_WB

import re
import time
from pathlib import Path

os.makedirs("intermediary", exist_ok=True)
os.makedirs("Graphics/NB1", exist_ok=True)

# Cache settings
# Each data source is saved to intermediary/cache/ after the first download.
# On subsequent runs, the cache is loaded directly, no API calls needed.
# Set FORCE_REFRESH = True to re-download everything from scratch.
FORCE_REFRESH = True
CACHE_DIR = Path("intermediary/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

import wbgapi as wb
import pandas as pd
from weo import download, WEO
import requests
import rdata


## 1. Variable Selection

All indicator codes and variable names are defined below for each source. This serves as the single point of reference: any variable added or removed from the analysis is changed here. For a full description of each variable's role, see Table 1 in the report.

In [29]:
# ──────────────────────────────────────────────────────────────
# World Bank Indicators
# ──────────────────────────────────────────────────────────────
wb_variables = [
    # Resource rents
    'NY.GDP.TOTL.RT.ZS',     # Total natural resources rents (% of GDP)
    'NY.GDP.MINR.RT.ZS',     # Mineral rents (% of GDP)
    'NY.GDP.NGAS.RT.ZS',     # Natural gas rents (% of GDP)
    'NY.GDP.PETR.RT.ZS',     # Oil rents (% of GDP)
    'NY.GDP.FRST.RT.ZS',     # Forestry rents (% of GDP)

    # Economic structure
    'NV.IND.MANF.ZS',        # Manufacturing, value added (% of GDP)
    'NV.IND.TOTL.ZS',        # Industry (incl. construction), value added (% of GDP)
    'TX.VAL.TECH.MF.ZS',     # High-technology exports (% of manufactured exports)
    'NV.AGR.TOTL.ZS',        # Agriculture, forestry, fishing, value added (% of GDP)
    'NV.SRV.TOTL.ZS',        # Services, value added (% of GDP)

    # Savings and capital
    'NY.ADJ.SVNG.CD',        # Adjusted savings: total (current US$)
    'NY.ADJ.ICTR.GN.ZS',     # Adjusted savings: gross savings (% of GNI)
    'NY.ADJ.DRES.GN.ZS',     # Adjusted savings: natural resources depletion (% of GNI)

    # Finance and debt
    'DT.DOD.DIMF.CD',        # Use of IMF credit (DOD, current US$)
    'FS.AST.PRVT.GD.ZS',     # Domestic credit to private sector (% of GDP)
    'FR.INR.RINR',            # Real interest rate (%)
    'FR.INR.LEND',            # Lending interest rate (%)
    'FP.CPI.TOTL.ZG',        # Inflation, consumer prices (annual %)

    # Employment
    'SL.IND.EMPL.ZS',        # Employment in industry (% of total employment)
    'SL.SRV.EMPL.ZS',        # Employment in services (% of total employment)
    'SL.AGR.EMPL.ZS',        # Employment in agriculture (% of total employment)

    # Infrastructure and demographics
    'EG.ELC.ACCS.ZS',        # Access to electricity (% of population)
    'IT.CEL.SETS.P2',        # Mobile cellular subscriptions (per 100 people)
    'SP.URB.TOTL',            # Urban population (% of total population)
    'SP.DYN.LE00.IN',        # Life expectancy at birth, total (years)
    'SP.DYN.CDRT.IN',        # Death rate, crude (per 1,000 people)

    # Trade
    'NE.TRD.GNFS.ZS',        # Trade (% of GDP)
]

# ──────────────────────────────────────────────────────────────
# IMF World Economic Outlook (subject, unit)
# ──────────────────────────────────────────────────────────────
imf_variables = [
    ("Gross domestic product per capita, constant prices",
     "Purchasing power parity; 2021 international dollar"),
    ("General government revenue",
     "Percent of GDP"),
    ("General government net debt",
     "Percent of GDP"),
    ("General government structural balance",
     "Percent of potential GDP"),
]

# ──────────────────────────────────────────────────────────────
# IMF Investment and Capital Stock Dataset (ICSD)
# Series codes for GFCF components and primary balance
# ──────────────────────────────────────────────────────────────
imf_icsd_variables = [
    'P51G_S13_Q_POGDP_PT.A',     # GFCF, General government, Constant prices, % of GDP
    'P51G_PS_Q_POGDP_PT.A',      # GFCF, Private sector, Constant prices, % of GDP
    'P51G_PUPVT_Q_POGDP_PT.A',   # GFCF, Public-private partnership, Constant prices, % of GDP
    'GPB_S13_POGDP_PT',          # Primary net lending, General government, % of GDP
]

# ──────────────────────────────────────────────────────────────
# Economic Complexity Index (dependent variable)
# ──────────────────────────────────────────────────────────────
eci_variables = ['eci']

# ──────────────────────────────────────────────────────────────
# V-Dem: Varieties of Democracy
# ──────────────────────────────────────────────────────────────
vdem_variables = [
    # High-level democracy indices
    'v2x_polyarchy',       # Electoral democracy index
    'v2x_libdem',          # Liberal democracy index
    'v2x_partipdem',       # Participatory democracy index
    'v2x_delibdem',        # Deliberative democracy index
    'v2x_egaldem',         # Egalitarian democracy index

    # Governance
    'v2xnp_client',        # Clientelism index
    'v2x_corr',            # Political corruption index
    'v2x_rule',            # Rule of law index
    'v2x_accountability',  # Accountability index
    'v2xcl_prpty',         # Property rights
    'e_wbgi_pve',          # Political stability (WGI estimate)
    'e_civil_war',         # Civil war indicator
]

# ──────────────────────────────────────────────────────────────
# Penn World Table 11.0
# Note: cn is in constant local currency; ctfp/cwtfp are indices
# ──────────────────────────────────────────────────────────────
pwt_variables = [
    'hc',     # Human capital index
    'cn',     # Capital stock (national accounts prices)
    'ctfp',   # TFP level (constant national prices, index)
    'cwtfp',  # Welfare-relevant TFP
    'csh_c',  # Share of consumption in GDP
    'csh_i',  # Share of investment in GDP
    'csh_g',  # Share of government spending in GDP
    'delta',  # Capital depreciation rate
]

# ──────────────────────────────────────────────────────────────
# CEPII
# ──────────────────────────────────────────────────────────────
cepii_variables = ['landlocked']

## 2. World Bank Indicators

Data is downloaded via the `wbgapi` Python package, which queries the World Bank API directly. We restrict the query to non-aggregate economies (i.e. real countries, excluding regional and income-group aggregates) and the 1995-2019 window.

In [30]:
def download_wb_indicators(indicators, start_year, end_year, max_retries=3, pause=2):
    """Download WBI indicators with retry logic for unstable API connections."""
    final_rows = []
    economies = [c['id'] for c in wb.economy.list() if not c.get("aggregate", False)]

    for indicator in indicators:
        for attempt in range(1, max_retries + 1):
            try:
                print(f"Downloading {indicator} ... (attempt {attempt})")
                raw = wb.data.fetch(indicator, economy=economies, time=range(start_year, end_year + 1))
                count = 0
                for row in raw:
                    iso   = row.get("economy")
                    year  = int(row.get("time").replace("YR", ""))
                    value = row.get("value")
                    if iso is None or value is None:
                        continue
                    final_rows.append({"Country Code": iso, "Year": year,
                                       "Variable": indicator, "Value": value})
                    count += 1
                print(f"  -> {count} rows")
                break
            except Exception as e:
                print(f"  Error: {e}")
                if attempt < max_retries:
                    wait = pause * attempt
                    print(f"  Retrying in {wait}s ...")
                    time.sleep(wait)
                else:
                    print(f"  FAILED after {max_retries} attempts. Skipping {indicator}.")
        time.sleep(1)

    return pd.DataFrame(final_rows)


_wb_cache = CACHE_DIR / 'wb_data.csv'

if not FORCE_REFRESH and _wb_cache.exists():
    wb_df = pd.read_csv(_wb_cache, dtype={'Country Code': str, 'Year': int})
    print(f"WB data loaded from cache: {wb_df.shape[0]:,} rows, {wb_df['Variable'].nunique()} indicators")
else:
    wb_df = download_wb_indicators(wb_variables, start_year=cfg.YEAR_MIN, end_year=cfg.YEAR_MAX)
    wb_df.to_csv(_wb_cache, index=False)
    print(f"\nWB data cached: {wb_df.shape[0]:,} rows, {wb_df['Variable'].nunique()} indicators")

  -> 5500 rows
  -> 5518 rows
  -> 5221 rows
  -> 5194 rows
  -> 5518 rows
  -> 5240 rows
  -> 5531 rows
  -> 2540 rows
  -> 5533 rows
  -> 5447 rows
  -> 3597 rows
  -> 4000 rows
  -> 4660 rows
  -> 3350 rows
  -> 4397 rows
  -> 3489 rows
  -> 3494 rows
  -> 5134 rows
  -> 5419 rows
  -> 5419 rows
  -> 5419 rows
  -> 5932 rows
  -> 5881 rows
  -> 6293 rows
  -> 6293 rows
  -> 6293 rows
  -> 5060 rows

WB data cached: 135,372 rows, 27 indicators


### Country name lookup table

We also build a lookup table mapping ISO3 codes to country names, which is used later to attach readable names to all sources.

In [31]:
_cn_cache = CACHE_DIR / 'country_names.csv'

if not FORCE_REFRESH and _cn_cache.exists():
    country_names = pd.read_csv(_cn_cache, dtype={'Country Code': str})
    print(f"Country lookup loaded from cache: {len(country_names)} entries")
else:
    all_economies = wb.economy.list()
    countries = [c for c in all_economies if not c.get("aggregate", False)]
    country_names = pd.DataFrame({
        "Country Code": [c["id"] for c in countries],
        "Country Name": [c["value"] for c in countries],
    })
    country_names.to_csv(_cn_cache, index=False)
    print(f"Country lookup cached: {len(country_names)} entries")

country_names.head()

Country lookup cached: 217 entries


,Country Code,Country Name
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,ALB,Albania
4,AND,Andorra


## 3. IMF World Economic Outlook

The `weo` Python package downloads the April 2024 vintage of the World Economic Outlook database. We extract four macroeconomic series: GDP per capita at PPP (constant 2017 international dollars), general government revenue (% of GDP), general government net debt (% of GDP), and the structural fiscal balance (% of potential GDP).

**Note on column naming:** The WEO `.get()` method returns a DataFrame with ISO3 codes as the index and years as columns. After reshaping to long format, the columns are renamed to match our standard schema (`Country Code`, `Year`, `Variable`, `Value`).

In [32]:
_weo_cache = CACHE_DIR / 'imf_weo.csv'

if not FORCE_REFRESH and _weo_cache.exists():
    imf_df = pd.read_csv(_weo_cache, dtype={'Country Code': str})
    imf_df['Year'] = imf_df['Year'].astype(int)
    print(f"IMF WEO loaded from cache: {imf_df.shape[0]:,} rows, {imf_df['Variable'].nunique()} indicators")
else:
    for attempt in range(1, 4):
        try:
            print(f"Downloading IMF WEO (attempt {attempt}) ...")
            # Try April 2025 vintage first, fall back to 2024 if not yet released
            try:
                path, _ = download(2025, "Apr")
                print("  Got April 2025 vintage.")
            except Exception:
                path, _ = download(2024, "Apr")
                print("  Got April 2024 vintage (2025 not yet available).")
            break
        except Exception as e:
            print(f"  Error: {e}")
            if attempt < 3:
                time.sleep(5)
            else:
                raise

    w = WEO(path)

    frames = []
    for subj, unit in imf_variables:
        # w.get() returns a DataFrame where the index is years and columns are ISO3 codes.
        # After reset_index(), years become a plain column (confusingly named 'index', then
        # renamed 'COUNTRY'). melt() then produces 'COUNTRY'=year values, 'YEAR'=country codes.
        # The final rename corrects this inversion to the standard schema.
        df = w.get(subj, unit).reset_index().rename(columns={"index": "COUNTRY"})
        df_long = df.melt(id_vars="COUNTRY", var_name="YEAR", value_name="VALUE")
        df_long["INDICATOR"] = subj
        frames.append(df_long)

    imf_df = pd.concat(frames, ignore_index=True)

    if os.path.exists("weo_2024_1.csv"):
        os.remove("weo_2024_1.csv")

    # Fix the inverted column names produced by the WEO DataFrame layout (see comment above)
    imf_df = imf_df.rename(columns={
        'COUNTRY':   'Year',
        'YEAR':      'Country Code',
        'INDICATOR': 'Variable',
        'VALUE':     'Value',
    })

    imf_df.to_csv(_weo_cache, index=False)
    print(f"IMF WEO cached: {imf_df.shape[0]:,} rows, {imf_df['Variable'].nunique()} indicators")

Already downloaded 2025-Apr WEO dataset at weo_2025_1.csv
  Got April 2025 vintage.
IMF WEO cached: 39,984 rows, 4 indicators


## 4. IMF Investment and Capital Stock Dataset (ICSD)

Two components are retrieved from the IMF ICSD, both hosted on the project's GitHub repository:

1. **Gross Fixed Capital Formation (GFCF):** Three sector-level series (general government, private sector, public-private partnerships) are summed into a single aggregate GFCF variable per country-year.
2. **Primary net lending:** General government primary balance as a share of GDP, from a separate ICSD file.

In [33]:
_icsd_cache = CACHE_DIR / 'imf_icsd.csv'

if not FORCE_REFRESH and _icsd_cache.exists():
    imf_icsd_df = pd.read_csv(_icsd_cache, dtype={'Country Code': str})
    imf_icsd_df['Year'] = imf_icsd_df['Year'].astype(int)
    print(f"IMF ICSD loaded from cache: {imf_icsd_df.shape[0]:,} rows")
else:
    # ── GFCF components (from IMF ICSD dataset) ──
    # The pattern matches only the 3 GFCF codes; GPB is in a different source file.
    gfcf_codes = [v for v in imf_icsd_variables if v.startswith('P51G')]
    pattern = "|".join(map(re.escape, gfcf_codes))

    _icsd_local = "/Users/leoss/Desktop/GitHub/leoss14.github.io/projects/capstone/New code/extension_2024/rawdata/dataset_2025-12-23T17_49_47.423426845Z_DEFAULT_INTEGRATION_IMF.FAD_ICSD_1.0.0.csv"
    _icsd_url   = ("https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/dataset_2025-12-23T17_49_47.423426845Z_DEFAULT_INTEGRATION_IMF.FAD_ICSD_1.0.0.csv")
    _icsd_src   = _icsd_local if os.path.exists(_icsd_local) else _icsd_url
    print(f"IMF ICSD source: {'local' if os.path.exists(_icsd_local) else 'GitHub URL'}")
    imf_icsd_df = (
        pd.read_csv(_icsd_src)
        .rename(columns={"COUNTRY": "Country Name", "INDICATOR": "Variable"})
        .loc[lambda df: df["SERIES_CODE"].str.contains(pattern, na=False)]
        .assign(Country_Code=lambda df: df["SERIES_CODE"].str[:3])
        .drop(columns=["DATASET", "SERIES_CODE", "OBS_MEASURE", "FREQUENCY", "SCALE"])
        .rename(columns={"Country_Code": "Country Code"})
    )

    # Intersect requested years with what the source actually contains.
    # The teammate's 2025-12-23 ICSD snapshot only goes to 2019; a fresher
    # IMF vintage would extend further. Either works without code changes.
    _requested_years = [str(y) for y in range(cfg.YEAR_MIN, cfg.YEAR_MAX + 1)]
    year_cols = [y for y in _requested_years if y in imf_icsd_df.columns]
    _missing = [y for y in _requested_years if y not in imf_icsd_df.columns]
    if _missing:
        print(f"  ICSD source missing years: {_missing[0]}-{_missing[-1]} "
              f"({len(_missing)} years). These will be NaN until a fresher vintage is pulled.")
    imf_icsd_df = pd.melt(
        imf_icsd_df,
        id_vars=["Country Code", "Variable"],
        value_vars=year_cols,
        var_name="Year",
        value_name="Value",
    )

    # Warn if any country-year has fewer than all 3 GFCF components (partial sum)
    component_counts = imf_icsd_df.groupby(["Country Code", "Year"])["Value"].count()
    incomplete = (component_counts < len(gfcf_codes)).sum()
    if incomplete > 0:
        print(f"  WARNING: {incomplete} country-year groups have fewer than "
              f"{len(gfcf_codes)} GFCF components, values will be partial sums")

    imf_icsd_df = (
        imf_icsd_df
        .groupby(["Country Code", "Year"], as_index=False)
        .agg({"Value": "sum"})
    )
    imf_icsd_df["Variable"] = "Gross fixed capital formation, all, Constant prices, Percent of GDP"

    # ── Primary net lending (from IMF FAD_FM dataset) ──
    _fad_local = "/Users/leoss/Desktop/GitHub/leoss14.github.io/projects/capstone/New code/extension_2024/rawdata/dataset_2026-01-05T21_29_20.743520144Z_DEFAULT_INTEGRATION_IMF.FAD_FM_5.0.0.csv"
    _fad_url   = ("https://raw.githubusercontent.com/AyaanTigdikar/Capstone/refs/heads/main/rawdata/dataset_2026-01-05T21_29_20.743520144Z_DEFAULT_INTEGRATION_IMF.FAD_FM_5.0.0.csv")
    _fad_src   = _fad_local if os.path.exists(_fad_local) else _fad_url
    print(f"IMF FAD_FM source: {'local' if os.path.exists(_fad_local) else 'GitHub URL'}")
    imf_icsd_df_2 = (
        pd.read_csv(_fad_src)
        .rename(columns={
            "COUNTRY.ID":   "Country Code",
            "INDICATOR.ID": "Variable",
            "TIME_PERIOD":  "Year",
            "OBS_VALUE":    "Value",
        })
        .drop(columns=["FREQUENCY.ID", "SCALE.ID"])
    )
    imf_icsd_df_2 = imf_icsd_df_2[
        (imf_icsd_df_2["Year"] >= cfg.YEAR_MIN) & (imf_icsd_df_2["Year"] <= cfg.YEAR_MAX)
    ]

    imf_icsd_df = pd.concat([imf_icsd_df, imf_icsd_df_2], ignore_index=True)
    imf_icsd_df.to_csv(_icsd_cache, index=False)
    print(f"IMF ICSD cached: {imf_icsd_df.shape[0]:,} rows")

IMF ICSD source: local
  ICSD source missing years: 2020-2023 (4 years). These will be NaN until a fresher vintage is pulled.
IMF FAD_FM source: local
IMF ICSD cached: 9,978 rows


## 5. Economic Complexity Index (ECI)

The Economic Complexity Index is our dependent variable. It measures how sophisticated and diversified a country's productive structure is by combining information on the diversity of exports a country produces and their ubiquity (i.e. the number of other countries able to produce them). A high ECI indicates that a country exports a wide range of products that few other countries can produce, reflecting deep embedded capabilities (Hausmann et al., 2014).

We source ECI values from the Atlas of Economic Complexity (HS92 classification).

In [34]:
_eci_cache = Path(cfg.CACHE) / 'eci.csv'

if not FORCE_REFRESH and _eci_cache.exists():
    eci_df = pd.read_csv(_eci_cache, dtype={'Country Code': str})
    eci_df['Year'] = eci_df['Year'].astype(int)
    print(f"ECI loaded from cache: {eci_df.shape[0]:,} rows, "
          f"{eci_df['Country Code'].nunique()} countries, "
          f"{eci_df['Year'].min()}-{eci_df['Year'].max()}, "
          f"variables: {sorted(eci_df['Variable'].unique())}")
else:
    # Read directly from the local Atlas file (downloaded by Leonardo from
    # atlas.cid.harvard.edu, May 2026). Schema: country_id, country_iso3_code,
    # year, growth_proj, in_rankings, eci_sitc, eci_rank_sitc, eci_hs92,
    # eci_rank_hs92, eci_hs12, eci_rank_hs12.
    eci_raw = pd.read_csv(cfg.ATLAS_FILE)
    print(f"Atlas raw: {eci_raw.shape[0]:,} rows, "
          f"{eci_raw['country_iso3_code'].nunique()} countries, "
          f"{eci_raw['year'].min()}-{eci_raw['year'].max()}")

    eci_col_name = f"eci_{cfg.ECI_CLASSIFICATION}"
    if eci_col_name not in eci_raw.columns:
        raise ValueError(f"Column {eci_col_name} not found in Atlas file. Available: {list(eci_raw.columns)}")

    # ECI series (used as the dependent variable downstream)
    eci_part = (
        eci_raw[["country_iso3_code", "year", eci_col_name]]
        .rename(columns={
            "country_iso3_code": "Country Code",
            "year":              "Year",
            eci_col_name:        "Value",
        })
        .assign(Variable="Economic Complexity")
    )

    parts = [eci_part]

    # Atlas growth projection (used as benchmark in the forecast notebook)
    if cfg.KEEP_GROWTH_PROJ and "growth_proj" in eci_raw.columns:
        growth_part = (
            eci_raw[["country_iso3_code", "year", "growth_proj"]]
            .dropna(subset=["growth_proj"])
            .rename(columns={
                "country_iso3_code": "Country Code",
                "year":              "Year",
                "growth_proj":       "Value",
            })
            .assign(Variable="Atlas growth projection")
        )
        parts.append(growth_part)
        print(f"  retained growth_proj: {len(growth_part):,} non-null country-years")

    eci_df = pd.concat(parts, ignore_index=True)[["Country Code", "Year", "Variable", "Value"]]

    # Clip to configured year range
    eci_df = eci_df[(eci_df["Year"] >= cfg.YEAR_MIN) & (eci_df["Year"] <= cfg.YEAR_MAX)]

    eci_df.to_csv(_eci_cache, index=False)
    print(f"ECI + growth_proj cached: {eci_df.shape[0]:,} rows, "
          f"{eci_df['Country Code'].nunique()} countries, "
          f"{eci_df['Year'].min()}-{eci_df['Year'].max()}, "
          f"variables: {sorted(eci_df['Variable'].unique())}")


Atlas raw: 4,319 rows, 145 countries, 1995-2024
  retained growth_proj: 2,683 non-null country-years
ECI + growth_proj cached: 6,714 rows, 145 countries, 1995-2023, variables: ['Atlas growth projection', 'Economic Complexity']


## 6. Varieties of Democracy (V-Dem)

The V-Dem dataset provides expert-coded measures of political institutions, governance, and democracy. We use indicators covering rule of law, property rights, political corruption, political stability, civil war, and five high-level democracy indices. The data is loaded from the `vdemdata` R package's `.RData` file hosted on GitHub.

In [35]:
_vdem_cache = CACHE_DIR / 'vdem.csv'

if not FORCE_REFRESH and _vdem_cache.exists():
    vdem_df = pd.read_csv(_vdem_cache, dtype={'Country Code': str})
    vdem_df['Year'] = vdem_df['Year'].astype(int)
    print(f"V-Dem loaded from cache: {vdem_df.shape[0]:,} rows, {vdem_df['Variable'].nunique()} indicators")
else:
    url = "https://raw.githubusercontent.com/vdeminstitute/vdemdata/master/data/vdem.RData"
    for attempt in range(1, 4):
        try:
            print(f"Downloading V-Dem .RData (~100MB, attempt {attempt}) ...")
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
            print("  Done.")
            break
        except Exception as e:
            print(f"  Error: {e}")
            if attempt < 3:
                time.sleep(10)
            else:
                raise

    with open("vdem.RData", "wb") as f:
        f.write(resp.content)

    try:
        vdem_r = rdata.read_rda("vdem.RData")
        vdem = vdem_r.get("vdem")
    finally:
        # Always clean up the temp file, even if rdata.read_rda raises
        if os.path.exists("vdem.RData"):
            os.remove("vdem.RData")

    var_list = ["country_name", "country_text_id", "year"] + vdem_variables
    vdem = vdem[var_list].rename(columns={
        "country_name":     "Country Name",
        "country_text_id":  "Country Code",
        "year":             "Year",
    })

    vdem_df = pd.melt(
        vdem,
        id_vars=["Country Code", "Year"],
        value_vars=vdem_variables,
        var_name="Variable",
        value_name="Value",
    )
    vdem_df = vdem_df[vdem_df["Year"] >= 1995]

    vdem_df.to_csv(_vdem_cache, index=False)
    print(f"V-Dem cached: {vdem_df.shape[0]:,} rows, {vdem_df['Variable'].nunique()} indicators")

  Done.


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/rdata/conversion/_conversion.py:856: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(


V-Dem cached: 66,168 rows, 12 indicators


## 7. Penn World Table 11.0

The Penn World Table provides internationally comparable macroeconomic indicators using purchasing power parity (PPP) adjustments. We extract human capital indices, capital stock, total factor productivity measures, expenditure shares, and the capital depreciation rate.

In [36]:
_pwt_cache = CACHE_DIR / 'pwt.csv'

if not FORCE_REFRESH and _pwt_cache.exists():
    pwt_df = pd.read_csv(_pwt_cache, dtype={'Country Code': str})
    pwt_df['Year'] = pwt_df['Year'].astype(int)
    print(f"PWT loaded from cache: {pwt_df.shape[0]:,} rows, {pwt_df['Variable'].nunique()} indicators")
else:
    # PWT 11.0 (release March 2025) covers 1950-2023 across every variable
    # listed in pwt_variables. Non-null counts per year through 2023:
    #   hc=145, cn=180, ctfp/cwtfp=120, csh_c/i/g=185, delta=180.
    # The 145/180/120 caps are country-coverage limits, not year limits.
    # Prefer local copy; fall back to the original GitHub mirror.
    local_pwt = "/Users/leoss/Desktop/GitHub/capstone-client-submission/main_analysis/rawdata/pwt110.xlsx"
    if os.path.exists(local_pwt):
        pwt_src = local_pwt
        print(f"PWT: reading local file {local_pwt}")
    else:
        pwt_src = "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/pwt110.xlsx"
        print(f"PWT: local file missing, falling back to GitHub mirror")

    pwt_df = (
        pd.read_excel(pwt_src, engine="openpyxl", sheet_name="Data")
        .rename(columns={"countrycode": "Country Code", "country": "Country Name", "year": "Year"})
    )
    pwt_df = pwt_df[["Country Code", "Country Name", "Year"] + pwt_variables]
    pwt_df = pwt_df.melt(
        id_vars=["Country Code", "Year"],
        value_vars=pwt_variables,
        var_name="Variable",
        value_name="Value",
    )
    pwt_df = pwt_df[(pwt_df["Year"] >= cfg.YEAR_MIN) & (pwt_df["Year"] <= cfg.YEAR_MAX)]
    pwt_df.to_csv(_pwt_cache, index=False)
    print(f"PWT cached: {pwt_df.shape[0]:,} rows, {pwt_df['Variable'].nunique()} indicators, "
          f"{pwt_df['Year'].min()}-{pwt_df['Year'].max()}")


PWT: reading local file /Users/leoss/Desktop/GitHub/capstone-client-submission/main_analysis/rawdata/pwt110.xlsx
PWT cached: 42,920 rows, 8 indicators, 1995-2023


## 8. CEPII: Landlocked Dummy

A binary indicator for whether a country is landlocked, from the CEPII GeoDist dataset. Landlocked countries face higher trade costs and limited access to maritime transport, which can constrain export diversification. Since this is a time-invariant characteristic, we expand it across all years in the panel.

In [37]:
_cepii_cache = CACHE_DIR / 'cepii.csv'

if not FORCE_REFRESH and _cepii_cache.exists():
    cepii_df = pd.read_csv(_cepii_cache, dtype={'Country Code': str})
    cepii_df['Year'] = cepii_df['Year'].astype(int)
    print(f"CEPII loaded from cache: {cepii_df.shape[0]:,} rows "
          f"({cepii_df['Country Code'].nunique()} countries x {cepii_df['Year'].nunique()} years)")
else:
    _cepii_local = "/Users/leoss/Desktop/GitHub/leoss14.github.io/projects/capstone/New code/extension_2024/rawdata/geo_cepii.xls"
    _cepii_url   = "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/geo_cepii.xls"
    _cepii_src   = _cepii_local if os.path.exists(_cepii_local) else _cepii_url
    print(f"CEPII source: {'local' if os.path.exists(_cepii_local) else 'GitHub URL'}")
    cepii_df = (
        pd.read_excel(_cepii_src, sheet_name="geo_cepii")
        .rename(columns={"iso3": "Country Code"})
        .drop_duplicates(subset=["Country Code"])
    )
    cepii_df = cepii_df[["Country Code"] + cepii_variables]
    cepii_df["Variable"] = "Landlocked"
    cepii_df = cepii_df.rename(columns={"landlocked": "Value"})
    # Expand to all years (time-invariant)
    years = pd.DataFrame({"Year": range(cfg.YEAR_MIN, cfg.YEAR_MAX + 1)})
    cepii_df = cepii_df.merge(years, how="cross")
    cepii_df.to_csv(_cepii_cache, index=False)
    print(f"CEPII cached: {cepii_df.shape[0]:,} rows "
          f"({cepii_df['Country Code'].nunique()} countries x {cepii_df['Year'].nunique()} years)")

CEPII source: local
CEPII cached: 6,525 rows (225 countries x 29 years)


## 9. Natural Resource Production Data

Production volumes for hydrocarbons (oil, natural gas, coal) and metals (lithium, cobalt, nickel, tin, bauxite, copper, aluminium, zinc, manganese, rare earth, platinum group, vanadium, natural graphite) are loaded from a pre-processed CSV. These are aggregated into two broad resource categories (Hydrocarbons and Metals) and summed by country-year.

Country names are standardised to match World Bank naming conventions to allow merging with the rest of the panel.

In [38]:
_prod_cache = CACHE_DIR / 'nr_production.csv'

if not FORCE_REFRESH and _prod_cache.exists():
    production = pd.read_csv(_prod_cache, dtype={'Country Code': str})
    production['Year'] = production['Year'].astype(int)
    print(f"NR production loaded from cache: {production.shape[0]:,} rows, "
          f"{production['Variable'].nunique()} resource-metric combinations")
else:
    # Local e0 output (preferred). Run e0_NR_extraction.ipynb to (re)produce.
    _e0_local = CACHE_DIR.parent / 'natural_resources_production_values.csv'

    if _e0_local.exists():
        print(f"NR production source: local e0 output ({_e0_local.name})")
        production = pd.read_csv(_e0_local)
        # e0 schema: Country, Year, Value, Resource, Metric, iso3
        # Rename to e1 schema: Country Name, Country Code
        production = production.rename(columns={
            'Country': 'Country Name',
            'iso3':    'Country Code',
        })
        # Drop rows where ISO3 mapping failed (regional aggregates, etc.)
        production = production[production['Country Code'].notna()]
        # WB-canonical country names (in case downstream merges expect them)
        production['Country Name'] = production['Country Code'].map(ISO3_TO_WB).fillna(production['Country Name'])
    else:
        # Fallback to teammate URL — this is known broken (404 at the time of
        # writing) but kept for compatibility if a working alternative emerges.
        print("WARNING: e0 output not found locally. Falling back to teammate URL.")
        production = (
            pd.read_csv(
                "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/refs/heads/main/"
                "../rawdata/production_values_w_prices-EM.csv"
            )
            .drop(columns="Unnamed: 0")
            .rename(columns={"Country": "Country Name"})
        )
        production = add_iso3(production, name_col="Country Name", iso3_col="Country Code")
        pre = len(production)
        production = production[production["Country Code"].notna()]
        print(f"Production: dropped {pre - len(production)} rows with unmatched country names")
        production["Country Name"] = production["Country Code"].map(ISO3_TO_WB)

    # Filter: drop consumption rows, clip to year range
    production = production[
        (production["Metric"] != "Consumption")
        & (production["Year"] >= cfg.YEAR_MIN)
        & (production["Year"] <= cfg.YEAR_MAX)
    ]

    # Categorize resources into Hydrocarbons / Metals / Others (matches v2 schema)
    hydrocarbons = ["Oil", "Natural Gas", "Coal"]
    metals = [
        "Lithium", "Cobalt", "Nickel", "Tin", "Bauxite", "Natural Graphite",
        "Copper", "Aluminium", "Zinc", "Manganese", "Rare Earth",
        "Platinum Group", "Vanadium",
    ]

    def classify_resource(x):
        if x in hydrocarbons:
            return "Hydrocarbons"
        elif x in metals:
            return "Metals"
        else:
            return "Others"

    production["Resource Category"] = production["Resource"].apply(classify_resource)
    production["Variable"] = production["Metric"] + "_" + production["Resource Category"]
    production = (
        production
        .groupby(["Country Code", "Year", "Variable"])["Value"]
        .sum()
        .reset_index()
    )

    production.to_csv(_prod_cache, index=False)
    print(f"NR production cached: {production.shape[0]:,} rows, "
          f"{production['Variable'].nunique()} resource-metric combinations, "
          f"{production['Year'].min()}-{production['Year'].max()}")


NR production source: local e0 output (natural_resources_production_values.csv)
NR production cached: 9,179 rows, 5 resource-metric combinations, 1995-2023


## 10. Merging and Final Cleaning

All seven source DataFrames are concatenated into a single long-format panel. Variable codes are then renamed to human-readable labels, and non-country territories (Hong Kong, Macao, Puerto Rico, etc.) are dropped.

In [39]:
# ── Concatenate all sources ──
final_df = pd.concat(
    [wb_df, eci_df, imf_df, imf_icsd_df, vdem_df, pwt_df, cepii_df, production],
    ignore_index=True,
)

# ── Rename variable codes to descriptive names ──
rename_map = {
    # World Bank
    "NV.IND.MANF.ZS":     "Manufacturing",
    "NV.IND.TOTL.ZS":     "Industry",
    "TX.VAL.TECH.MF.ZS":  "High-tech exports",
    "NV.AGR.TOTL.ZS":     "Agriculture",
    "NV.SRV.TOTL.ZS":     "Services",
    "NY.GDP.TOTL.RT.ZS":  "Total natural resources rents (% of GDP)",
    "NY.GDP.MINR.RT.ZS":  "Mineral rents (% of GDP)",
    "NY.GDP.NGAS.RT.ZS":  "Natural gas rents (% of GDP)",
    "NY.GDP.PETR.RT.ZS":  "Oil rents (% of GDP)",
    "NY.GDP.FRST.RT.ZS":  'Forestry rents (% of GDP)',
    "NY.ADJ.SVNG.CD":     "Adjusted savings: total (current US$)",
    "NY.ADJ.ICTR.GN.ZS":  "Adjusted savings: gross savings (% of GNI)",
    "NY.ADJ.DRES.GN.ZS":  "Adjusted savings: natural resources depletion (% of GNI)",
    "DT.DOD.DIMF.CD":     "Use of IMF credit (DOD, current US$)",
    "SL.IND.EMPL.ZS":     "Employment in industry (% of total employment)",
    "SL.SRV.EMPL.ZS":     "Employment in services (% of total employment)",
    "SL.AGR.EMPL.ZS":     "Employment in agriculture (% of total employment)",
    "EG.ELC.ACCS.ZS":     "Access to electricity (% of population)",
    "IT.CEL.SETS.P2":     "Mobile cellular subscriptions (per 100 people)",
    "FS.AST.PRVT.GD.ZS":  "Domestic credit to private sector (% of GDP)",
    "FR.INR.RINR":         "Real interest rate (%)",
    "FR.INR.LEND":         "Lending interest rate (%)",
    "FP.CPI.TOTL.ZG":     "Inflation, consumer prices (annual %)",
    "SP.URB.TOTL":         "Urban population (% of total population)",
    "SP.DYN.LE00.IN":     "Life expectancy at birth, total (years)",
    "NE.TRD.GNFS.ZS":     "Trade (% of GDP)",
    "SP.DYN.CDRT.IN":     "Death rates, crude per 1000 people",

    # ECI
    "Economic Complexity": "Economic Complexity Index",

    # IMF WEO
    "Gross domestic product per capita, constant prices": "GDP per capita (constant prices, PPP)",
    "General government revenue": "Government revenue",

    # IMF ICSD
    "GPB_S13_POGDP_PT": "Primary net lending, General government, Percent of GDP",

    # V-Dem
    "v2x_polyarchy":      "electoral_dem",
    "v2x_libdem":         "liberal_dem",
    "v2x_partipdem":      "participatory_dem",
    "v2x_delibdem":       "deliberative_dem",
    "v2x_egaldem":        "egalitarian_dem",
    "v2xnp_client":       "Clientelism index",
    "v2x_corr":           "Political corruption index",
    "v2x_rule":           "Rule of law index",
    "v2x_accountability": "Accountability index",
    "v2xcl_prpty":        "Property rights",
    "e_wbgi_pve":         "Political stability — estimate",
    "e_civil_war":        "Civil war",

    # PWT
    "hc":    "Human capital index",
    "cn":    "Capital stock (national accounts prices)",
    "ctfp":  "TFP level (constant national prices)",
    "cwtfp": "Welfare-relevant TFP",
    "csh_c": "Share of consumption in GDP",
    "csh_i": "Share of investment in GDP",
    "csh_g": "Share of government spending in GDP",
    "delta": "Capital depreciation rate",

    # CEPII
    "landlocked": "Landlocked",
}

final_df["Variable"] = final_df["Variable"].replace(rename_map)

# ── Attach country names ──
final_df = final_df.merge(country_names, how="left", on="Country Code")

# ── Standardise Year column ──
# Some sources return pandas Period objects; convert everything to int
final_df["Year"] = final_df["Year"].apply(
    lambda x: x.year if hasattr(x, "year") else int(x)
)
final_df = final_df[(final_df["Year"] >= cfg.YEAR_MIN) & (final_df["Year"] <= cfg.YEAR_MAX)]

# ── Remove non-country territories ──
not_countries = [
    "HKG", "MAC", "PRI", "VIR", "GUM", "ASM", "CYM", "BMU",
    "GRL", "MAF", "SXM", "CUW", "ABW", "FRO", "MNP", "PYF",
]
final_df = final_df[~final_df["Country Code"].isin(not_countries)]

print(f"Merged dataset: {final_df.shape[0]:,} rows")
print(f"Countries: {final_df['Country Code'].nunique()}")
print(f"Variables: {final_df['Variable'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")

Merged dataset: 286,287 rows
Countries: 251
Variables: 61
Years: 1995 - 2023


## 11. Variable Selection for Analysis

Not all downloaded variables are used in the final econometric and ML analysis. The list below flags the subset of variables retained for modelling. This selection is based on theoretical relevance, data coverage, and preliminary correlation analysis (see report Section 3.4).

In [40]:
# Variables retained for the econometric and ML pipeline
important_vars = [
    # Governance
    "Rule of law index",
    "Political stability — estimate",
    "Property rights",

    # Macroeconomic
    "GDP per capita (constant prices, PPP)",
    "Government revenue",
    "Inflation, consumer prices (annual %)",
    "Lending interest rate (%)",
    "Real interest rate (%)",
    "Share of consumption in GDP",
    "Share of government spending in GDP",
    "Share of investment in GDP",

    # GDP structure
    "Economic Complexity Index",
    "Agriculture",
    "Industry",
    "Manufacturing",
    "Services",

    # Finance
    "Adjusted savings: gross savings (% of GNI)",
    "Gross fixed capital formation, all, Constant prices, Percent of GDP",
    "Capital depreciation rate",

    # Human capital and infrastructure
    "Access to electricity (% of population)",
    "Human capital index",
    "Life expectancy at birth, total (years)",
    "Mobile cellular subscriptions (per 100 people)",
    "Urban population (% of total population)",

    # Resource rents
    "Mineral rents (% of GDP)",
    "Natural gas rents (% of GDP)",
    "Oil rents (% of GDP)",
    "Forestry rents (% of GDP)",
    "Total natural resources rents (% of GDP)",
]

final_df["Important_vars"] = final_df["Variable"].isin(important_vars).astype(int)
print(f"Variables flagged as important: {len(important_vars)}")
print(f"Total unique variables in dataset: {final_df['Variable'].nunique()}")

Variables flagged as important: 29
Total unique variables in dataset: 61


## 12. Reshape to Wide Format and Export

The long-format panel is pivoted to wide format (one row per country-year, one column per variable) for use in regression and ML notebooks. Both formats are saved.

In [41]:
print(f"Unique variables before pivot: {final_df['Variable'].nunique()}")

final_df_wide = final_df.pivot(
    index=["Country Code", "Country Name", "Year"],
    columns="Variable",
    values="Value",
).reset_index()

print(f"Wide dataset shape: {final_df_wide.shape}")
print(f"  = {final_df_wide['Country Code'].nunique()} countries "
      f"x {final_df_wide['Year'].nunique()} years "
      f"+ {final_df_wide.shape[1] - 3} variables")

Unique variables before pivot: 61
Wide dataset shape: (7234, 64)
  = 251 countries x 29 years + 61 variables


## 13. Save Outputs

In [42]:
final_df.to_csv("intermediary/master_data_long.csv", index=False)
final_df_wide.to_csv("intermediary/master_data_wide.csv", index=False)

# Headline panel for downstream consumers. Kept under a distinct name from
# the live pipeline's intermediary/Master.csv to avoid accidental cross-talk.
final_df_wide.to_csv("intermediary/Master_extended.csv", index=False)

print("Saved:")
print("  intermediary/master_data_long.csv")
print("  intermediary/master_data_wide.csv")
print("  intermediary/Master_extended.csv  (headline; downstream notebooks read this)")


Saved:
  intermediary/master_data_long.csv
  intermediary/master_data_wide.csv
  intermediary/Master_extended.csv  (headline; downstream notebooks read this)


---

## Summary

| Stage | Description |
|-------|-------------|
| **Sources** | World Bank (26 indicators), IMF WEO (4), IMF ICSD (2 aggregated), ECI (1), V-Dem (12), PWT (8), CEPII (1), Production (resource-metric combos) |
| **Panel** | 1995-2019, all non-aggregate World Bank economies minus 16 non-country territories |
| **Output** | `master_data_long.csv` (long format), `master_data_wide.csv` (wide format, one row per country-year) |
| **Next step** | Missingness diagnostics (`MissingCheck.ipynb`) and imputation (`3_Imputing.ipynb`) |

**Pipeline position:** This is Step 1 of 3 in the data preparation sequence:  
`1_cleaning_master_data.ipynb` → `MissingCheck.ipynb` (diagnostics) → `3_Imputing.ipynb` (sample selection + imputation) → Analysis notebooks

In [43]:
# ── NB1 SUMMARY ──
print("=" * 70)
print("NB1: DATA CONSTRUCTION SUMMARY")
print("=" * 70)
print(f"\nLong format: {final_df.shape[0]:,} rows")
print(f"Wide format: {final_df_wide.shape}")
print(f"Countries: {final_df['Country Code'].nunique()}")
print(f"Variables: {final_df['Variable'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")
print(f"\nVariables flagged as important: {final_df['Important_vars'].sum():,} rows")
print(f"\nWide format columns:")
for col in sorted(final_df_wide.columns):
    if col not in ['Country Code', 'Country Name', 'Year']:
        miss = final_df_wide[col].isna().sum()
        print(f"  {col}: {miss} missing ({100*miss/len(final_df_wide):.1f}%)")
print(f"\nFiles saved:")
print(f"  intermediary/master_data_long.csv")
print(f"  intermediary/master_data_wide.csv")

NB1: DATA CONSTRUCTION SUMMARY

Long format: 286,287 rows
Wide format: (7234, 64)
Countries: 251
Variables: 61
Years: 1995 - 2023

Variables flagged as important: 145,542 rows

Wide format columns:
  Access to electricity (% of population): 1737 missing (24.0%)
  Accountability index: 2107 missing (29.1%)
  Adjusted savings: gross savings (% of GNI): 3346 missing (46.3%)
  Adjusted savings: natural resources depletion (% of GNI): 2607 missing (36.0%)
  Adjusted savings: total (current US$): 3637 missing (50.3%)
  Agriculture: 1920 missing (26.5%)
  Atlas growth projection: 4698 missing (64.9%)
  Capital depreciation rate: 2159 missing (29.8%)
  Capital stock (national accounts prices): 2159 missing (29.8%)
  Civil war: 5286 missing (73.1%)
  Clientelism index: 2107 missing (29.1%)
  Death rates, crude per 1000 people: 1405 missing (19.4%)
  Domestic credit to private sector (% of GDP): 2924 missing (40.4%)
  Economic Complexity Index: 3089 missing (42.7%)
  Employment in agriculture (%

## 14. Coverage diagnostic

Per-variable non-null counts by year, focused on the post-2019 window. Use this to decide whether to keep the extended panel unbalanced (default), drop year-variable combinations with thin coverage, or upgrade specific sources (e.g. PWT 11.0 -> 11.1 for HCI).


In [44]:
# Per-variable coverage by year, post-2019 window
important_set = set(important_vars)
core = final_df_wide.copy()
year_range = sorted(core['Year'].unique())
post_2019_years = [y for y in year_range if y >= 2019]

# Build a (variable x year) matrix of non-null counts
rows = []
for v in sorted(important_set & set(core.columns)):
    counts = {y: int(core.loc[core['Year'] == y, v].notna().sum()) for y in post_2019_years}
    counts['variable'] = v
    rows.append(counts)

import pandas as pd
cov = pd.DataFrame(rows).set_index('variable')
cov = cov[post_2019_years]  # column order

print(f'Total countries in panel: {core["Country Code"].nunique()}')
print()
print('Non-null country count per (variable, year), post-2019 window:')
print(cov.to_string())
print()

# Flag variables whose 2024 coverage is < 50% of 2019 coverage
if 2019 in cov.columns and 2024 in cov.columns:
    weak = cov[(cov[2024] < 0.5 * cov[2019]) & (cov[2019] > 0)].copy()
    if len(weak) > 0:
        weak['ratio_2024_vs_2019'] = (weak[2024] / weak[2019]).round(2)
        print('Variables with weak post-2019 coverage (2024 < 50% of 2019):')
        print(weak[[2019, 2024, 'ratio_2024_vs_2019']].to_string())
    else:
        print('All important variables maintain >=50% of 2019 coverage through 2024.')


Total countries in panel: 251

Non-null country count per (variable, year), post-2019 window:
                                                                     2019  2020  2021  2022  2023
variable                                                                                         
Access to electricity (% of population)                               200   200   200   200   200
Adjusted savings: gross savings (% of GNI)                            154   151   130     0     0
Agriculture                                                           190   189   189   187   183
Capital depreciation rate                                             175   175   175   175   175
Economic Complexity Index                                             144   144   144   144   144
Forestry rents (% of GDP)                                             193   191   186     0     0
GDP per capita (constant prices, PPP)                                 189   189   189   189   190
Government revenue      